In [69]:
"""PII detection module for Legal Document Review System.

This module provides functionality to detect Personally Identifiable Information (PII)
in text using regex patterns and context analysis for confidence scoring.
"""

import re
from dataclasses import dataclass
from typing import List, Tuple


@dataclass
class PIIDetection:
    """Represents a detected PII instance."""
    pii_type: str  # 'ssn', 'email', 'phone', 'credit_card'
    value: str  # The detected PII value
    start_pos: int  # Start position in text
    end_pos: int  # End position in text
    confidence: float  # Confidence score (0.0 to 1.0)


class PIIDetector:
    """
    Detects Personally Identifiable Information (PII) in text using pattern matching
    and context analysis.

    Detects:
    - SSN (Social Security Numbers): XXX-XX-XXXX format
    - Email addresses: user@domain.com format
    - Phone numbers: (XXX) XXX-XXXX or XXX-XXX-XXXX format
    - Credit card numbers: 16-digit card numbers

    Uses context analysis to calculate confidence scores for detected PII.
    """

    def __init__(self):
        """
        Initializes the PIIDetector with PII detection patterns.
        """
        # Define PII patterns
        self.pii_patterns = {
            'ssn': r'\b\d{3}-\d{2}-\d{4}\b',
            'email': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
            'phone': r'\b\d{3}-\d{3}-\d{4}\b|\b\(\d{3}\)\s?\d{3}-\d{4}\b',
            'credit_card': r'\b(?:4\d{3}|5[1-5]\d{2})[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b'
        }

        # Context indicators that boost confidence
        self.context_indicators = {
            'ssn': ['social security', 'ssn', 'ss#', 'tax id'],
            'email': ['email', 'contact', 'e-mail', 'mail'],
            'phone': ['phone', 'mobile', 'call', 'telephone', 'contact'],
            'credit_card': ['card', 'payment', 'credit', 'visa', 'mastercard']
        }

    def _calculate_confidence(
        self,
        pii_type: str,
        match_text: str,
        surrounding_context: str,
    ) -> float:
        """
        Calculate confidence score for detected PII based on contextual indicators.

        Args:
            pii_type: Type of PII detected ('ssn', 'email', 'phone', 'credit_card').
            match_text: The matched PII text.
            surrounding_context: Text surrounding the detected match.

        Returns:
            Confidence score between 0.0 and 1.0.
        """
        base_confidence = 0.7

        if not match_text or not surrounding_context:
            return round(base_confidence, 1)

        context = surrounding_context.casefold()
        indicators = self.context_indicators.get(pii_type, [])

        for indicator in indicators:
            if indicator and indicator.casefold() in context:
                base_confidence += 0.1

        return min(round(base_confidence, 1), 1.0)

    def detect_pii(self, text: str) -> List[PIIDetection]:
        """
        Detects all PII instances in the given text.

        Process:
        1. Search for each PII pattern in the text
        2. For each match, extract surrounding context
        3. Calculate confidence score based on context
        4. Return list of PIIDetection objects

        Args:
            text (str): The text to scan for PII.

        Returns:
            List[PIIDetection]: List of detected PII instances with positions and confidence scores.

        Raises:
            TypeError: If text is not a string.
            ValueError: If text is empty.
        """
        if not isinstance(text, str):
            raise TypeError("text must be a string")

        if not text.strip():
            raise ValueError("text cannot be empty")

        results = []

        for pii_type, pattern in self.pii_patterns.items():
            for match in re.finditer(pattern, text):
                start_pos, end_pos = match.span()
                surrounding_context = self._get_surrounding_context(text, start_pos, end_pos)
                confidence = self._calculate_confidence(pii_type, match.group(), surrounding_context)
                results.append(PIIDetection(
                    pii_type=pii_type,
                    value=match.group(),
                    start_pos=start_pos,
                    end_pos=end_pos,
                    confidence=confidence
                ))

        return results

    def _get_surrounding_context(self, text: str, start_pos: int, end_pos: int, context_window: int = 50) -> str:
        """
        Extracts surrounding context around a detected PII.

        Args:
            text (str): The full text.
            start_pos (int): Start position of PII.
            end_pos (int): End position of PII.
            context_window (int): Number of characters before and after to include (default: 50).

        Returns:
            str: Surrounding context string.
        """
        if not isinstance(text, str):
            raise TypeError("text must be a string")

        if not text.strip():
            raise ValueError("text cannot be empty")

        start_context = max(0, start_pos - context_window)
        end_context = min(len(text), end_pos + context_window)
        return text[start_context:end_context]


In [49]:
"""PII masking module for Legal Document Review System.

This module provides functionality to mask detected PII according to enterprise
policies while preserving partial information where appropriate.
"""

from typing import List

# from src.pii_detector import PIIDetection


class PIIMasker:
    """
    Masks Personally Identifiable Information (PII) in text according to enterprise policies.

    Masking Strategies:
    - SSN: XXX-XX-1234 (preserve last 4 digits)
    - Email: j***h@company.com (mask username, preserve domain)
    - Phone: ***-***-4567 (preserve last 4 digits)
    - Credit Card: **** **** **** 9012 (show last 4 digits only)
    """

    def __init__(self):
        """
        Initializes the PIIMasker.
        """
        self.detector = PIIDetector()

    def mask_ssn(self, ssn: str) -> str:
        """
        Masks a Social Security Number, preserving last 4 digits.

        Args:
            ssn (str): SSN in format XXX-XX-XXXX.

        Returns:
            str: Masked SSN in format XXX-XX-1234.

        Raises:
            ValueError: If SSN format is invalid.
        """
        self.detector.pii_patterns.get('ssn')  # Ensure SSN pattern is defined
        if not re.match(self.detector.pii_patterns['ssn'], ssn):
            raise ValueError("Invalid SSN format")

        return f"XXX-XX-{ssn.split('-')[-1]}"

    def mask_email(self, email: str) -> str:
        """
        Masks an email address, preserving domain.

        Args:
            email (str): Email address (e.g., "john.doe@company.com").

        Returns:
            str: Masked email (e.g., "j***e@company.com" or "j***d@company.com").

        Raises:
            ValueError: If email format is invalid.
        """
        self.detector.pii_patterns.get('email')  # Ensure email pattern is defined
        if not re.match(self.detector.pii_patterns['email'], email):
            raise ValueError("Invalid email format")

        username, domain = email.split('@')
        masked_username = username[0] + '*' * 3 + username[-1]

        return f"{masked_username}@{domain}"

    def mask_phone(self, phone: str) -> str:
        """
        Masks a phone number, preserving last 4 digits.

        Args:
            phone (str): Phone number in format XXX-XXX-XXXX or (XXX) XXX-XXXX.

        Returns:
            str: Masked phone (e.g., "***-***-4567").

        Raises:
            ValueError: If phone format is invalid.
        """
        self.detector.pii_patterns.get('phone')  # Ensure phone pattern is defined
        clean_phone = phone.replace('(', '').replace(')', '').replace(' ', '-')
        print(clean_phone)
        if not re.match(self.detector.pii_patterns['phone'], clean_phone):
            raise ValueError("Invalid phone format")

        return f"***-***-{clean_phone.split('-')[-1]}"

    def mask_credit_card(self, credit_card: str) -> str:
        """
        Masks a credit card number, showing only last 4 digits.

        Args:
            credit_card (str): Credit card number (may include spaces or dashes).

        Returns:
            str: Masked credit card (e.g., "**** **** **** 9012").

        Raises:
            ValueError: If credit card format is invalid.
        """
        self.detector.pii_patterns.get('credit_card')  # Ensure credit card pattern is defined
        if not re.match(self.detector.pii_patterns['credit_card'], credit_card):
            raise ValueError("Invalid credit card format")

        return f"**** **** **** {credit_card.split()[-1]}"

    def mask_text(self, text: str, pii_detections: List[PIIDetection]) -> str:
        """
        Masks all detected PII in text according to their types.

        Process:
        1. Sort detections by position (reverse order to avoid position shifts)
        2. For each detection, apply appropriate masking based on type
        3. Replace PII in text with masked version

        Args:
            text (str): The text containing PII.
            pii_detections (List[PIIDetection]): List of detected PII instances.

        Returns:
            str: Text with all PII masked.

        Raises:
            TypeError: If text is not a string or pii_detections is not a list.
        """
        if not isinstance(text, str):
            raise TypeError("text must be a string")

        if not isinstance(pii_detections, list):
            raise TypeError("pii_detections must be a list")

        # Sort detections by start position (reverse order)
        pii_detections.sort(key=lambda x: x.start_pos, reverse=True)

        for detection in pii_detections:
            if detection.pii_type == 'ssn':
                masked_value = self.mask_ssn(detection.value)
            elif detection.pii_type == 'email':
                masked_value = self.mask_email(detection.value)
            elif detection.pii_type == 'phone':
                masked_value = self.mask_phone(detection.value)
            elif detection.pii_type == 'credit_card':
                masked_value = self.mask_credit_card(detection.value)
            else:
                continue  # Unknown PII type

            # Replace original PII with masked version in text
            text = text[:detection.start_pos] + masked_value + text[detection.end_pos:]

        return text


In [50]:
"""Content filtering module for Legal Document Review System.

This module provides functionality to filter content for safety violations,
privileged information, and policy compliance.
"""

from dataclasses import dataclass
from typing import Dict, List


@dataclass
class ContentSafetyResult:
    """Result of content safety evaluation."""
    safety_score: float  # Score from 0.0 to 1.0 (higher = safer)
    violations: List[str]  # List of detected violations
    requires_disclaimer: bool  # Whether content requires disclaimer
    is_safe: bool  # Whether content is safe to return (score >= 0.8)


class ContentFilter:
    """
    Filters content for safety violations and privileged information.

    Detects:
    - Prohibited content categories (violence, discrimination, illegal activities)
    - Privileged/confidential information indicators
    - Sensitive domains requiring disclaimers (legal advice, medical advice)

    Calculates safety scores and identifies violations.
    """

    def __init__(self):
        """
        Initializes the ContentFilter with prohibited content patterns.
        """
        # Prohibited content categories
        self.prohibited_keywords = {
            'violence': ['harm', 'attack', 'hurt', 'violence', 'threat', 'assault'],
            'discrimination': ['exclude', 'discriminate', 'bias', 'prejudice', 'racist'],
            'illegal': ['fraud', 'illegal', 'contraband', 'criminal', 'unlawful']
        }

        # Sensitive domains requiring disclaimers
        self.sensitive_domains = [
            'legal advice',
            'medical advice',
            'financial guidance',
            'tax advice',
            'investment recommendation'
        ]

        # Privileged information indicators
        self.privileged_indicators = [
            'attorney-client privilege',
            'confidential communication',
            'privileged and confidential',
            'work product'
        ]

    def evaluate_safety(self, text: str) -> ContentSafetyResult:
        """
        Evaluates content safety and identifies violations.

        Process:
        1. Check for prohibited content categories
        2. Check for sensitive domains requiring disclaimers
        3. Check for privileged information indicators
        4. Calculate safety score (start at 1.0, deduct for violations)
        5. Determine if content is safe to return (score >= 0.8)

        Args:
            text (str): The text to evaluate.

        Returns:
            ContentSafetyResult: Safety evaluation result with score, violations, and flags.

        Raises:
            TypeError: If text is not a string.
            ValueError: If text is empty.
        """
        if not isinstance(text, str):
            raise TypeError("text must be a string")

        if not text.strip():
            raise ValueError("text cannot be empty")

        prohibited_content = self._check_prohibited_content(text)
        sensitive_domains = self._check_sensitive_domains(text)
        privileged_info = self._check_privileged_information(text)

        # Calculate safety score
        safety_score = 1.0
        if prohibited_content:
            safety_score -= 0.2 * len(prohibited_content)
        if sensitive_domains:
            safety_score -= 0.1 * len(sensitive_domains)
        if privileged_info:
            safety_score -= 0.3

        safety_score = max(0.0, min(safety_score, 1.0))
        is_safe = safety_score >= 0.8
        requires_disclaimer = bool(sensitive_domains)

        return ContentSafetyResult(
            safety_score=safety_score,
            violations=list(set(prohibited_content + sensitive_domains + (['privileged information'] if privileged_info else []))),
            requires_disclaimer=requires_disclaimer,
            is_safe=is_safe
        )

    def _check_prohibited_content(self, text: str) -> List[str]:
        """
        Checks for prohibited content categories.

        Args:
            text (str): The text to check.

        Returns:
            List[str]: List of detected prohibited categories.
        """
        unique_words = set(word.lower() for word in re.findall(r'\b\w+\b', text))
        detected_categories = []

        for category, keywords in self.prohibited_keywords.items():
            if unique_words.intersection(keywords):
                detected_categories.append(category)

        return detected_categories

    def _check_sensitive_domains(self, text: str) -> List[str]:
        """
        Checks for sensitive domains requiring disclaimers.

        Args:
            text (str): The text to check.

        Returns:
            List[str]: List of detected sensitive domains.
        """
        detected_domains = []

        for domain in self.sensitive_domains:
            if domain in text.lower():
                detected_domains.append(domain)

        return detected_domains

    def _check_privileged_information(self, text: str) -> bool:
        """
        Checks if text contains privileged information indicators.

        Args:
            text (str): The text to check.

        Returns:
            bool: True if privileged information indicators are found.
        """
        for privileged in self.privileged_indicators:
            if privileged.lower() in text.lower():
                return True


In [51]:
"""RAG generator module for Legal Document Review System.

This module provides functionality to generate answers from retrieved documents.
"""

from typing import Dict, List


class RAGGenerator:
    """
    A class for generating answers from retrieved documents.

    This is a simplified generator that creates answers by combining
    information from retrieved documents.
    """

    def __init__(self):
        """
        Initializes the RAGGenerator.
        """
        pass

    def generate_answer(self, query: str, retrieved_documents: List[Dict]) -> str:
        """
        Generates an answer from retrieved documents.

        Args:
            query (str): The user's query.
            retrieved_documents (List[Dict]): List of retrieved documents with 'content' key.

        Returns:
            str: Generated answer combining information from retrieved documents.
        """
        if not retrieved_documents:
            return "I could not find relevant information to answer your query."

        # Simple answer generation: combine content from top documents
        answer_parts = []
        answer_parts.append(f"Based on the retrieved documents:\n\n")

        for i, doc in enumerate(retrieved_documents[:3], 1):  # Use top 3 documents
            content = doc.get('content', '')
            if content:
                answer_parts.append(f"{i}. {content[:200]}...")  # Truncate for brevity

        return "\n".join(answer_parts)



In [52]:
"""Vector store module for semantic search in Legal Document Review System.

This module provides functionality to create, manage, and query vector embeddings
for semantic search using sentence transformers.
"""

import os
import pickle
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


class VectorStore:
    """
    A class for creating and managing vector embeddings and indexes for semantic search.
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initializes the VectorStore with a specified embedding model.

        Args:
            model_name (str): Name of the SentenceTransformer model to use.
                Default: "all-MiniLM-L6-v2" (384 dimensions)
        """
        self.model_name = model_name
        self.model = None

    def _ensure_model_loaded(self):
        """Lazy load the SentenceTransformer model."""
        if self.model is None:
            self.model = SentenceTransformer(self.model_name)

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generates sentence embeddings for a list of texts using SentenceTransformer.

        Args:
            texts (List[str]): A list of text strings.

        Returns:
            np.ndarray: Array of sentence embedding vectors (shape: [n_texts, embedding_dim]).
        """
        if not isinstance(texts, list):
            raise TypeError("texts must be a list")

        if len(texts) == 0:
            raise ValueError("texts list cannot be empty")

        self._ensure_model_loaded()
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        return embeddings

    def create_index(self, df: pd.DataFrame, index_file_name: str, index_folder_name: str) -> Dict:
        """
        Creates a vector index from the given DataFrame and saves it to disk.

        Args:
            df (pd.DataFrame): DataFrame containing documents with required fields.
            index_file_name (str): Name of the index file.
            index_folder_name (str): Directory where the index will be saved.

        Returns:
            dict: The created index containing embeddings and metadata.
        """
        if not isinstance(df, pd.DataFrame):
            raise ValueError("df must be a pandas DataFrame")

        if len(df) == 0:
            raise ValueError("DataFrame is empty")

        required_columns = ['id', 'title', 'content', 'category']
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")

        texts = [f"{row['title']}\n\n{row['content']}" for _, row in df.iterrows()]
        embeddings = self.generate_embeddings(texts)

        index = {
            'embeddings': embeddings,
            'id': df['id'].tolist(),
            'title': df['title'].tolist(),
            'content': df['content'].tolist(),
            'category': df['category'].tolist()
        }

        index_folder = Path(index_folder_name)
        index_folder.mkdir(parents=True, exist_ok=True)

        index_path = index_folder / index_file_name
        with open(index_path, 'wb') as f:
            pickle.dump(index, f)

        return index

    def load_index(self, index_file_name: str, index_folder_name: str) -> Dict:
        """
        Loads a precomputed vector index from disk.

        Args:
            index_file_name (str): The file name of the saved index.
            index_folder_name (str): The directory where the index is stored.

        Returns:
            dict: The loaded index containing embeddings and metadata.
        """
        index_path = Path(index_folder_name) / index_file_name

        if not index_path.exists():
            raise FileNotFoundError(f"Index file not found: {index_path}")

        try:
            with open(index_path, 'rb') as f:
                index = pickle.load(f)
        except Exception as e:
            raise ValueError(f"Failed to load index: {str(e)}")

        required_keys = ['embeddings', 'id', 'title', 'content', 'category']
        missing_keys = [key for key in required_keys if key not in index]
        if missing_keys:
            raise KeyError(f"Index missing required keys: {missing_keys}")

        if not isinstance(index['embeddings'], np.ndarray):
            raise ValueError("Index embeddings must be a numpy array")

        if len(index['embeddings']) == 0:
            raise ValueError("Index embeddings array is empty")

        n_docs = len(index['embeddings'])
        for key in required_keys:
            if len(index[key]) != n_docs:
                raise ValueError(f"Index metadata length mismatch: {key} has {len(index[key])} items, expected {n_docs}")

        return index

    def get_query_embedding(self, query: str) -> np.ndarray:
        """
        Generates an embedding for a single query string.

        Args:
            query (str): The query string to embed.

        Returns:
            np.ndarray: The embedding vector for the query (shape: [embedding_dim]).
        """
        if not isinstance(query, str):
            raise ValueError("query must be a string")

        if not query.strip():
            raise ValueError("query cannot be empty")

        self._ensure_model_loaded()
        embedding = self.model.encode(query, convert_to_numpy=True)

        if embedding.ndim > 1:
            embedding = embedding[0]

        return embedding

    def find_top_k_matches(self, query_embedding: np.ndarray, index: Dict, k: int = 5) -> List[Tuple[int, float]]:
        """
        Finds the top-k most similar entries from the index using cosine similarity.

        Args:
            query_embedding (np.ndarray): The embedding of the input query.
            index (dict): The index containing document embeddings and metadata.
            k (int): The number of top matches to return (default: 5).

        Returns:
            list: A list of tuples (document_index, similarity_score) for top-k matches.
        """
        if not isinstance(query_embedding, np.ndarray):
            raise TypeError("query_embedding must be a numpy array")

        if 'embeddings' not in index:
            raise ValueError("Index must contain 'embeddings' key")

        if not isinstance(index['embeddings'], np.ndarray):
            raise ValueError("Index embeddings must be a numpy array")

        doc_embeddings = index['embeddings']

        if query_embedding.ndim != 1:
            raise ValueError("query_embedding must be 1D array")

        if doc_embeddings.ndim != 2:
            raise ValueError("doc_embeddings must be 2D array")

        if query_embedding.shape[0] != doc_embeddings.shape[1]:
            raise ValueError(f"Dimension mismatch: query has {query_embedding.shape[0]} dims, docs have {doc_embeddings.shape[1]} dims")

        query_norm = np.linalg.norm(query_embedding)
        doc_norms = np.linalg.norm(doc_embeddings, axis=1)

        dot_products = np.dot(doc_embeddings, query_embedding)
        similarities = dot_products / (query_norm * doc_norms)
        similarities = np.nan_to_num(similarities, nan=0.0)

        top_k_indices = np.argsort(similarities)[::-1][:k]
        results = [(int(idx), float(similarities[idx])) for idx in top_k_indices]

        return results


In [ ]:
"""Secure RAG system integrating PII detection, masking, and content filtering.

This module provides a secure RAG pipeline that:
1. Retrieves relevant documents.
2. Generates an answer.
3. Detects PII in the generated answer.
4. Masks detected PII.
5. Evaluates the masked answer for content safety.
6. Returns the secured response with metadata.
"""

from __future__ import annotations

from typing import Any

# from src.content_filter import ContentFilter, ContentSafetyResult
# from src.pii_detector import PIIDetector
# from src.pii_masker import PIIMasker
# from src.rag_generator import RAGGenerator
# from src.vector_store import VectorStore


class SecureRAGSystem:
    """Secure RAG system integrating retrieval, generation, and security."""

    def __init__(
        self,
        vector_store: VectorStore,
        rag_generator: RAGGenerator,
        pii_detector: PIIDetector,
        pii_masker: PIIMasker,
        content_filter: ContentFilter,
        index: dict[str, Any],
    ) -> None:
        """
        Initialize the secure RAG system.

        Args:
            vector_store: Vector store used for similarity search.
            rag_generator: Component responsible for generating answers.
            pii_detector: Component responsible for detecting PII.
            pii_masker: Component responsible for masking PII.
            content_filter: Component responsible for content safety.
            index: Pre-loaded vector index.
        """
        self.vector_store = vector_store
        self.rag_generator = rag_generator
        self.pii_detector = pii_detector
        self.pii_masker = pii_masker
        self.content_filter = content_filter
        self.index = index

    def search(
        self,
        query: str,
        k: int = 5,
    ) -> dict[str, Any]:
        """
        Execute a secure RAG search.

        Args:
            query: User's search query.
            k: Maximum number of documents to retrieve.

        Returns:
            Dictionary containing:
                - answer
                - safety_result
                - pii_detected
                - source_documents

        Raises:
            TypeError: If query is not a string or k is not an integer.
            ValueError: If query is empty or k is not positive.
        """
        self._validate_search_parameters(query, k)

        retrieved_docs = self._retrieve_documents(query, k)

        generated_answer = self.rag_generator.generate_answer(
            query,
            retrieved_docs,
        )

        (
            masked_answer,
            safety_result,
            pii_count,
        ) = self._apply_security_measures(generated_answer)

        source_documents = [
            document["id"]
            for document in retrieved_docs
        ]

        return self._format_response(
            masked_answer=masked_answer if retrieved_docs else generated_answer,
            safety_result=safety_result,
            pii_count=pii_count,
            source_documents=source_documents,
        )

    @staticmethod
    def _validate_search_parameters(
        query: str,
        k: int,
    ) -> None:
        """Validate search parameters."""
        if not isinstance(query, str):
            raise TypeError("query must be a string")

        if not query.strip():
            raise ValueError("query cannot be empty")

        # bool is technically an int subclass, so explicitly reject it.
        if isinstance(k, bool) or not isinstance(k, int):
            raise TypeError("k must be an integer")

        if k <= 0:
            raise ValueError("k must be a positive integer")

    def _retrieve_documents(
        self,
        query: str,
        k: int,
    ) -> list[dict[str, Any]]:
        """
        Retrieve the top-k documents for a query.
        """
        query_embedding = (
            self.vector_store.get_query_embedding(query)
        )

        retrieved_docs = (
            self.vector_store.find_top_k_matches(
                query_embedding,
                self.index,
                k,
            )
        )

        result_docs: list[dict[str, Any]] = []

        for doc_idx, similarity in retrieved_docs:
            result_docs.append(
                {
                    "id": self.index["id"][doc_idx],
                    "content": self.index["content"][doc_idx],
                    "similarity": similarity,
                }
            )

        return result_docs

    def _apply_security_measures(
        self,
        answer: str,
    ) -> tuple[str, ContentSafetyResult, int]:
        """
        Detect and mask PII, then evaluate content safety.
        """
        pii_detections = self.pii_detector.detect_pii(answer)

        pii_count = len(pii_detections)

        masked_answer = self.pii_masker.mask_text(
            answer,
            pii_detections,
        )

        safety_result = (
            self.content_filter.evaluate_safety(
                masked_answer
            )
        )

        return (
            masked_answer,
            safety_result,
            pii_count,
        )

    @staticmethod
    def _format_response(
        masked_answer: str,
        safety_result: ContentSafetyResult,
        pii_count: int,
        source_documents: list[str],
    ) -> dict[str, Any]:
        """Format the final secure response."""
        return {
            "answer": masked_answer,
            "safety_result": {
                "safety_score": safety_result.safety_score,
                "violations": safety_result.violations,
                "requires_disclaimer": (
                    safety_result.requires_disclaimer
                ),
                "is_safe": safety_result.is_safe,
            },
            "pii_detected": pii_count,
            "source_documents": source_documents,
        }


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2030.23it/s]
